# S06 · Limpieza de inconsistencias
## NovaMarket · Ricardo Borja

Estandarización de categorías de texto, conversión de `fecha_pedido` a tipo fecha y
verificación de duplicados sobre el dataset ya normalizado.

Insumo: `S04_PD_Borja_DatasetImputacionInicial.csv`, salida de la sesión de valores faltantes.
Salida: `S06_PD_Borja_DatasetDepurado.csv`.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

ENTRADA = "S04_PD_Borja_DatasetImputacionInicial.csv"
CRUDO   = "NovaMarket_datos_crudos.csv"
SALIDA  = "S06_PD_Borja_DatasetDepurado.csv"

CATEGORICAS = ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]

df = pd.read_csv(ENTRADA, dtype=str)
filas_inicio = len(df)

pd.DataFrame({"filas": [filas_inicio], "columnas": [df.shape[1]]}, index=["dataset de entrada"])

,filas,columnas
dataset de entrada,620000,27


---
## 1 · Diagnóstico previo

Conteo de variantes por columna antes de modificar nada.

In [2]:
diagnostico = pd.DataFrame({
    "variantes_encontradas": [df[c].nunique(dropna=True) for c in CATEGORICAS],
    "nulos": [int(df[c].isna().sum()) for c in CATEGORICAS],
}, index=CATEGORICAS)
diagnostico

,variantes_encontradas,nulos
ciudad_tienda,12,0
categoria_producto,32,0
canal_compra,20,0
metodo_pago,20,0


In [3]:
for c in CATEGORICAS:
    print(f"{c} ({df[c].nunique(dropna=True)} variantes)")
    print(sorted(df[c].dropna().unique()))
    print()

ciudad_tienda (12 variantes)
['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']

categoria_producto (32 variantes)
[' Belleza', ' Deportes', ' Electrónica', ' Hogar', ' Juguetería', ' Moda', 'BELLEZA', 'Belleza', 'Belleza ', 'DEPORTES', 'Deportes', 'Deportes ', 'ELECTRÓNICA', 'Electrónica', 'Electrónica ', 'HOGAR', 'Hogar', 'Hogar ', 'JUGUETERÍA', 'Juguetería', 'Juguetería ', 'MODA', 'Moda', 'Moda ', 'belleza', 'deportes', 'electronica', 'electrónica', 'hogar', 'jugueteria', 'juguetería', 'moda']

canal_compra (20 variantes)
[' App', ' Marketplace', ' Tienda', ' Web', 'APP', 'App', 'App ', 'MARKETPLACE', 'Marketplace', 'Marketplace ', 'TIENDA', 'Tienda', 'Tienda ', 'WEB', 'Web', 'Web ', 'app', 'marketplace', 'tienda', 'web']

metodo_pago (20 variantes)
[' Efectivo', ' PayPal', ' Tarjeta', ' Transferencia', 'EFECTIVO', 'Efectivo', 'Efectivo ', 'PAYPAL', 'PayPal', 'PayPal ', 'TARJETA', 'TR

Las variantes provienen de tres fuentes: espacios al inicio o al final, diferencias de
mayúsculas y minúsculas, y ausencia de tildes. `ciudad_tienda` ya llega con una sola forma por
ciudad porque fue imputada en la sesión anterior.

---
## 2 · Estandarización de texto

Primera pasada con `.str.strip().str.title()`, que resuelve espacios y mayúsculas.

In [4]:
base = {c: df[c].str.strip().str.title() for c in CATEGORICAS}

pd.DataFrame({
    "antes": [df[c].nunique(dropna=True) for c in CATEGORICAS],
    "tras_strip_title": [base[c].nunique(dropna=True) for c in CATEGORICAS],
}, index=CATEGORICAS)

,antes,tras_strip_title
ciudad_tienda,12,12
categoria_producto,32,8
canal_compra,20,4
metodo_pago,20,4


In [5]:
for c in CATEGORICAS:
    print(f"{c}: {sorted(base[c].dropna().unique())}")

ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Electrónica', 'Hogar', 'Jugueteria', 'Juguetería', 'Moda']
canal_compra: ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: ['Efectivo', 'Paypal', 'Tarjeta', 'Transferencia']


### Diccionario de corrección manual

`.str.title()` no resuelve dos casos, porque opera sobre la forma de las letras y no sobre su
significado. Quedan pendientes las variantes de abajo, que se agrupan con un mapeo explícito.

| Columna | Variante | Se agrupa en | Criterio |
|---|---|---|---|
| `categoria_producto` | `Electronica` | `Electrónica` | La escritura sin tilde es un error de digitación, no una categoría distinta. `.title()` no puede agregar la tilde que falta. |
| `categoria_producto` | `Jugueteria` | `Juguetería` | Mismo caso. La forma con tilde es la correcta en español y la que ya usa la mayoría de los registros. |
| `metodo_pago` | `Paypal` | `PayPal` | `.title()` fuerza mayúscula solo en la primera letra y daña la grafía de la marca. La forma con P mayúscula intermedia es la oficial. |

Las tres agrupaciones se eligen conservando la forma que ya predomina en el dataset, de modo que
el mapeo corrige la minoría mal escrita y no reescribe la mayoría.

In [6]:
MAPA_CORRECCION = {
    "categoria_producto": {"Electronica": "Electrónica", "Jugueteria": "Juguetería"},
    "metodo_pago": {"Paypal": "PayPal"},
}

for c in CATEGORICAS:
    df[c] = base[c].replace(MAPA_CORRECCION.get(c, {}))

for c in CATEGORICAS:
    print(f"{c} ({df[c].nunique(dropna=True)} categorías)")
    print(sorted(df[c].dropna().unique()))
    print()

ciudad_tienda (12 categorías)
['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']

categoria_producto (6 categorías)
['Belleza', 'Deportes', 'Electrónica', 'Hogar', 'Juguetería', 'Moda']

canal_compra (4 categorías)
['App', 'Marketplace', 'Tienda', 'Web']

metodo_pago (4 categorías)
['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']



Control: la estandarización no puede crear ni destruir valores vacíos.

In [7]:
control_nulos = pd.DataFrame({
    "nulos_antes": diagnostico["nulos"],
    "nulos_despues": [int(df[c].isna().sum()) for c in CATEGORICAS],
}, index=CATEGORICAS)
control_nulos["diferencia"] = control_nulos["nulos_despues"] - control_nulos["nulos_antes"]
control_nulos

,nulos_antes,nulos_despues,diferencia
ciudad_tienda,0,0,0
categoria_producto,0,0,0
canal_compra,0,0,0
metodo_pago,0,0,0


También se recortan espacios en las columnas de texto libre que no son categóricas, para que
no arrastren diferencias invisibles a la comparación de duplicados.

In [8]:
TEXTO_LIBRE = ["producto", "comentario_cliente", "nivel_satisfaccion", "nivel_lealtad"]
for c in TEXTO_LIBRE:
    if c in df.columns:
        df[c] = df[c].str.strip()

pd.DataFrame({"columnas_recortadas": [c for c in TEXTO_LIBRE if c in df.columns]})

,columnas_recortadas
0,producto
1,comentario_cliente
2,nivel_satisfaccion
3,nivel_lealtad


---
## 3 · Conversión de `fecha_pedido`

Inventario de formatos presentes en la columna.

In [9]:
texto_fecha = df["fecha_pedido"].str.strip()

es_iso    = texto_fecha.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
es_barras = texto_fecha.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)
es_vacia  = texto_fecha.isna()

formatos = pd.DataFrame({
    "filas": [int(es_iso.sum()), int(es_barras.sum()), int(es_vacia.sum()),
              int((~es_iso & ~es_barras & ~es_vacia).sum())],
}, index=["ISO aaaa-mm-dd", "con barras d/m/aaaa", "vacías", "sin patrón reconocido"])
formatos

,filas
ISO aaaa-mm-dd,199608
con barras d/m/aaaa,414858
vacías,5534
sin patrón reconocido,0


El formato con barras es ambiguo entre día/mes y mes/día. La prueba siguiente resuelve cuál es:
si el segundo campo nunca supera 12 y el primero sí, entonces el primero es el día.

In [10]:
partes = texto_fecha[es_barras].str.split("/", expand=True).astype(int)

pd.DataFrame({
    "maximo": [int(partes[0].max()), int(partes[1].max())],
    "filas_mayores_a_12": [int((partes[0] > 12).sum()), int((partes[1] > 12).sum())],
}, index=["primer campo", "segundo campo"])

,maximo,filas_mayores_a_12
primer campo,31,249965
segundo campo,12,0


El primer campo llega a 31 y el segundo nunca pasa de 12, así que el orden es día/mes/año y la
conversión se hace con `dayfirst=True`. Cada formato se convierte por separado, en lugar de dejar
que pandas infiera, para que ninguna fecha se interprete con el orden equivocado en silencio.

In [11]:
nulos_fecha_antes = int(df["fecha_pedido"].isna().sum())

fecha = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
fecha[es_iso]    = pd.to_datetime(texto_fecha[es_iso], format="%Y-%m-%d", errors="coerce")
fecha[es_barras] = pd.to_datetime(texto_fecha[es_barras], dayfirst=True, errors="coerce")

df["fecha_pedido_invalida"] = fecha.isna() & ~es_vacia
df["fecha_pedido"] = fecha

pd.DataFrame({
    "valor": [str(df["fecha_pedido"].dtype),
              str(df["fecha_pedido"].min().date()),
              str(df["fecha_pedido"].max().date())],
}, index=["tipo de dato", "fecha mínima", "fecha máxima"])

,valor
tipo de dato,datetime64[ns]
fecha mínima,2025-01-01
fecha máxima,2027-12-28


Los valores vacíos suben. El detalle de abajo muestra que el aumento corresponde por completo a
fechas de calendario imposible, no a errores de conversión.

In [12]:
imposibles = texto_fecha[df["fecha_pedido_invalida"]].value_counts().rename("filas")
imposibles.to_frame()

,filas
fecha_pedido,
31/04/2025,5513
30/02/2025,5507
2026-13-05,5446


In [13]:
nulos_fecha_despues = int(df["fecha_pedido"].isna().sum())

pd.DataFrame({"filas": [
    nulos_fecha_antes,
    int(df["fecha_pedido_invalida"].sum()),
    nulos_fecha_despues,
    nulos_fecha_despues - nulos_fecha_antes - int(df["fecha_pedido_invalida"].sum()),
]}, index=["vacías antes de convertir",
           "fechas de calendario imposible",
           "vacías después de convertir",
           "aumento sin explicar"])

,filas
vacías antes de convertir,5534
fechas de calendario imposible,16466
vacías después de convertir,22000
aumento sin explicar,0


Las fechas imposibles se marcan en `fecha_pedido_invalida` y no se eliminan: la fila conserva
el resto de sus datos de compra, que siguen siendo válidos. La marca permite excluirlas de
cualquier análisis temporal sin perder el registro.

---
## 4 · Duplicados

La verificación se hace sobre el dataset ya normalizado. Dos filas que solo diferían en
`Moda ` contra `moda` no se detectaban como duplicadas antes de la estandarización, así que
contar duplicados sobre el texto crudo habría subestimado el problema.

Se revisan cuatro definiciones, de la más estricta a la más laxa.

In [14]:
sin_id = [c for c in df.columns if c != "id_pedido"]
CLAVE_LOGICA = ["id_cliente", "fecha_pedido", "producto", "monto_compra", "ciudad_tienda"]

duplicados = pd.DataFrame({"filas_duplicadas": [
    int(df.duplicated().sum()),
    int(df.duplicated(subset=sin_id).sum()),
    int(df.duplicated(subset=CLAVE_LOGICA).sum()),
    int(df["id_pedido"].duplicated().sum()),
]}, index=["exactos en las 27 columnas",
           "idénticos ignorando id_pedido",
           "misma compra lógica (cliente, fecha, producto, monto, ciudad)",
           "id_pedido repetido"])
duplicados

,filas_duplicadas
exactos en las 27 columnas,0
idénticos ignorando id_pedido,0
"misma compra lógica (cliente, fecha, producto, monto, ciudad)",0
id_pedido repetido,0


### Duplicados por identificador de cliente

`correo_cliente` es el identificador que pide revisar el enunciado. El conteo directo es
engañoso y se descompone abajo.

In [15]:
CENTINELA = "desconocido"
correo_real = df.loc[df["correo_cliente"] != CENTINELA, "correo_cliente"]

pd.DataFrame({"filas": [
    int(df["correo_cliente"].duplicated().sum()),
    int((df["correo_cliente"] == CENTINELA).sum()),
    int(correo_real.duplicated().sum()),
]}, index=["correo repetido, conteo directo",
           "filas con el centinela 'desconocido'",
           "correo repetido, excluyendo el centinela"])

,filas
"correo repetido, conteo directo",101999
filas con el centinela 'desconocido',102000
"correo repetido, excluyendo el centinela",0


Las 101.999 filas que aparecen como correo repetido son el centinela `desconocido` que la
sesión anterior escribió donde el correo faltaba. No son el mismo cliente: son 102.000 pedidos
distintos cuyo correo se desconoce.

Descontando el centinela, ningún correo se repite. Un `drop_duplicates(subset="correo_cliente")`
aplicado sin esta distinción eliminaría 101.999 pedidos legítimos.

### Regla de conservación

No hay duplicados que eliminar bajo ninguna de las cuatro definiciones, así que la regla no
retira ninguna fila. Se deja aplicada de todos modos para que quede fijada si el dataset se
reprocesa con datos nuevos.

**Regla.** Ante filas idénticas en todas las columnas salvo `id_pedido`, se conserva la de
`fecha_actualizacion_stock` más reciente, y ante empate la primera en aparecer. El motivo es que
esa columna indica cuándo se refrescó el registro por última vez, de modo que la copia más
reciente es la que refleja el estado vigente del inventario para ese pedido.

In [16]:
antes_regla = len(df)

df = (df.sort_values("fecha_actualizacion_stock", ascending=False, kind="mergesort")
        .drop_duplicates(subset=sin_id, keep="first")
        .sort_index())

pd.DataFrame({"filas": [antes_regla, len(df), antes_regla - len(df)]},
             index=["antes de la regla", "después de la regla", "eliminadas"])

,filas
antes de la regla,620000
después de la regla,620000
eliminadas,0


Contexto de trazabilidad: el archivo crudo tenía 640.000 filas con 20.000 duplicados exactos.
La sesión de valores faltantes los eliminó y entregó las 620.000 filas que son el insumo de este
notebook. Por eso el conteo de arriba da cero y no porque la verificación haya fallado.

In [17]:
crudo = pd.read_csv(CRUDO, usecols=["id_pedido"], dtype=str)

pd.DataFrame({"filas": [
    len(crudo),
    int(crudo.duplicated().sum()),
    filas_inicio,
    len(crudo) - int(crudo["id_pedido"].duplicated().sum()) - filas_inicio,
]}, index=["crudo original",
           "id_pedido repetido en el crudo",
           "insumo de este notebook",
           "diferencia sin explicar"])

,filas
crudo original,640000
id_pedido repetido en el crudo,20000
insumo de este notebook,620000
diferencia sin explicar,0


---
## 5 · Verificación final y exportación

In [18]:
verificacion = pd.DataFrame({
    "categorias_finales": [df[c].nunique(dropna=True) for c in CATEGORICAS],
    "nulos": [int(df[c].isna().sum()) for c in CATEGORICAS],
}, index=CATEGORICAS)
verificacion

,categorias_finales,nulos
ciudad_tienda,12,0
categoria_producto,6,0
canal_compra,4,0
metodo_pago,4,0


In [19]:
for c in CATEGORICAS:
    print(f"{c}: {sorted(df[c].dropna().unique())}")

ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
categoria_producto: ['Belleza', 'Deportes', 'Electrónica', 'Hogar', 'Juguetería', 'Moda']
canal_compra: ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']


In [20]:
resumen = pd.DataFrame({"valor": [
    filas_inicio,
    len(df),
    df.shape[1],
    str(df["fecha_pedido"].dtype),
    int(df["fecha_pedido_invalida"].sum()),
    int(df["fecha_pedido"].isna().sum()),
    int(df.duplicated().sum()),
]}, index=["filas de entrada",
           "filas de salida",
           "columnas de salida",
           "tipo de fecha_pedido",
           "fechas marcadas como inválidas",
           "fecha_pedido sin valor",
           "duplicados exactos restantes"])
resumen

,valor
filas de entrada,620000
filas de salida,620000
columnas de salida,28
tipo de fecha_pedido,datetime64[ns]
fechas marcadas como inválidas,16466
fecha_pedido sin valor,22000
duplicados exactos restantes,0


In [21]:
df.to_csv(SALIDA, index=False)

pd.DataFrame({"valor": [SALIDA, f"{os.path.getsize(SALIDA)/1048576:.1f} MB", len(df)]},
             index=["archivo", "tamaño", "filas"])

,valor
archivo,S06_PD_Borja_DatasetDepurado.csv
tamaño,141.2 MB
filas,620000
